# Single SWV Trace Fit

This notebook shows the smallest ASWIFT workflow: load one voltage/current trace, run `aswift_fit` and `poly_linear_fit`, and plot the fitted peak and background profiles.

The notebook uses the first current trace from the public ASWIFT example-data archive. If the archive is not already extracted locally, the setup cell downloads and unzips it automatically.

In [ ]:
from pathlib import Path
import sys
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import pandas as pd

# Needed only when running from a source checkout instead of an installed package.
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "src" / "aswift").exists():
        repo_root = candidate
        break
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from aswift import AswiftSettings, aswift_fit, poly_linear_fit
from aswift import plot_fit_result

## Load Example Data

The example archive is stored in the public ASWIFT repository. The download is skipped when the extracted example file is already available.

In [ ]:
EXAMPLE_DATA_URL = "https://github.com/Soh-Lab/aswift/releases/download/v1.0.3/example_data.zip"
examples_dir = repo_root / "examples" if (repo_root / "examples").exists() else Path.cwd().resolve()
bundled_archive = repo_root / "release_assets" / "example_data.zip"
EXAMPLE_DATA_ZIP = bundled_archive if bundled_archive.exists() else examples_dir / "example_data.zip"
DATA_PATH = examples_dir / "example_data" / "simple_csv_example" / "250hz-1.csv"

if not DATA_PATH.exists():
    if not EXAMPLE_DATA_ZIP.exists():
        urllib.request.urlretrieve(EXAMPLE_DATA_URL, EXAMPLE_DATA_ZIP)
    with zipfile.ZipFile(EXAMPLE_DATA_ZIP) as zf:
        zf.extractall(examples_dir)

df = pd.read_csv(DATA_PATH)
volts = df.iloc[:, 0].to_numpy(dtype=float)
current = df.iloc[:, 1].to_numpy(dtype=float)

## Fit With ASWIFT and Poly-Linear

In [ ]:
aswift_result = aswift_fit(volts, current)
poly_result = poly_linear_fit(volts, current)

pd.DataFrame([
    aswift_result.to_record(),
    poly_result.to_record(),
])[["method", "peak", "background", "peak_voltage", "success", "error"]]

## Plot The Fits

`FitResult` stores full-length `peak_profile`, `background_profile`, and `fitted_current`, so plotting does not require reconstructing model coefficients.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
plot_fit_result(aswift_result, ax=axes[0])
axes[0].set_title("ASWIFT")
plot_fit_result(poly_result, ax=axes[1])
axes[1].set_title("Poly-linear")
plt.show()

## Change ASWIFT Settings

ASWIFT defaults are chosen to work without manual tuning, but the settings object lets you inspect or deliberately change behavior. A common adjustment is `peak_prominence`, which controls how wide the prominence-based peak window is before local Tikhonov peak fitting. Larger values, such as `1.0`, use a wider peak region; smaller values focus the local fit more tightly around the peak.

Other settings:

- `baseline_boundary`: fraction of points protected at each edge when identifying the dominant peak window.
- `bg_buffer`: extra samples added to both sides of the detected peak window.
- `huber_reweight`: enables robust residual weighting during smoothing and peak fitting.
- `huber_cutoff`: scaled-residual cutoff for Huber-style downweighting; lower values downweight more aggressively.
- `mad_window`: rolling window used to estimate the local residual scale for robust weights.
- `peak_lambda_scale`: multiplier applied to the selected peak-smoothing lambda. Values below `1` allow a more flexible peak profile; values above `1` smooth more strongly.
- `peak_prominence`: relative-height threshold used to define the local peak window.


In [ ]:
default_settings = AswiftSettings()
wide_peak_settings = AswiftSettings(peak_prominence=1.0)

default_aswift = aswift_fit(volts, current, settings=default_settings)
wide_peak_aswift = aswift_fit(volts, current, settings=wide_peak_settings)

pd.DataFrame([
    {"settings": "default", **default_aswift.to_record()},
    {"settings": "peak_prominence=1.0", **wide_peak_aswift.to_record()},
])[["settings", "peak", "background", "peak_voltage", "success", "error"]]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
plot_fit_result(default_aswift, ax=axes[0])
axes[0].set_title("Default ASWIFT")
plot_fit_result(wide_peak_aswift, ax=axes[1])
axes[1].set_title("ASWIFT, peak_prominence=1.0")
plt.show()
